# Code Graph Workbench — App to Foundation Integration Architecture

Companion to `2026-06-09-code-graph-workbench-app-design.md`. Where the `.md` defines **what** the app is, this notebook details the **agent-driven** integration with the foundation / core notebook runtime.

**Model B1 (agent-driven):** the user asks the app-owned ACP agent; during its turn the agent calls **app-level evidence tools** (an MCP plugin server in `server/`, the `html_video` model) that run deterministic queries, serialize Arrow, and `notebook_push_source` the results onto declared source ports. Those pushes cascade into the AFM panels. The same turn streams the answer.

| Layer | Members |
|---|---|
| **App** (`app.ipynb`) | `wb-question`, `wb-graph`, `wb-analyst`, `wb-inspector` (frontend) + `wb-answer` (AI node) |
| **App plugin** (`server/`) | FastMCP server: `wb_blast_radius`, `wb_subgraph`, `wb_scorecard`, `wb_cochange` |
| **Foundation** (jute-notebook daemon) | ReactiveEngine, Port Store, Notebook MCP server (+ proxied plugin tools), AFM host, AiNodeBackend runner |
| **Services / artifacts** | `.spur/analyst.duckdb` (DuckPGQ + Onager views), `code_*` tools, `spur-acp` `NativeAcpConnection` + agent subprocess |

> **Invariant:** evidence tools run REAL queries server-side; the LLM only chooses arguments. Every painted node/row is real, never hallucinated.

## 1. System context

Four zones: sandboxed app cells, the app's own MCP plugin, the trusted foundation daemon, and external services / artifacts. The app never touches the orchestration brain; its only agent path is an app-owned ACP session.

```mermaid
flowchart LR
  User([Developer])
  subgraph Z1["App cells (sandboxed)"]
    APP["app.ipynb"]
  end
  subgraph Z2["App plugin (server/, FastMCP)"]
    SRV["wb_* evidence tools"]
  end
  subgraph Z3["Foundation daemon (trusted)"]
    FND["ReactiveEngine + Port Store<br/>Notebook MCP + AFM host"]
  end
  subgraph Z4["Services / artifacts"]
    AD[".spur/analyst.duckdb"]
    GR["code_* tools"]
    AC["spur-acp NativeAcpConnection"]
    AGT(["Agent subprocess"])
  end
  User -->|ask| APP
  APP -->|AI node turn| AC
  AC -->|spawn + prompt| AGT
  AGT -->|call wb_* via notebook MCP socket| FND
  FND -->|proxy plugin tools| SRV
  SRV -->|read-only| AD
  SRV -->|code_*| GR
  SRV -->|notebook_push_source| FND
  FND -->|cascade render| APP
  APP -->|streamed answer| User
```

## 2. Integration architecture (agent → app tools → push cascade)

The centerpiece. The agent reaches the app's `wb_*` tools through the notebook MCP socket (which proxies plugin tools additively); the tools query, build Arrow, and push back through `notebook_push_source`, which queues the ReactiveEngine and cascades to the AFM widgets.

```mermaid
flowchart TB
  subgraph App["Code Graph Workbench app.ipynb"]
    Q["wb-question<br/>(frontend)"]
    G["wb-graph<br/>(AFM)"]
    A["wb-analyst<br/>(AFM)"]
    I["wb-inspector<br/>(AFM)"]
    AN["wb-answer<br/>(AI node)"]
  end
  subgraph Plugin["App MCP plugin (server/, FastMCP, foundation-spawned)"]
    T1["wb_blast_radius"]
    T2["wb_subgraph"]
    T3["wb_scorecard"]
    T4["wb_cochange"]
  end
  subgraph Foundation["Foundation / core notebook (daemon)"]
    RE["ReactiveEngine"]
    PS["Port Store<br/>Arrow + manifest.json"]
    MCP["Notebook MCP server<br/>SPUR_NOTEBOOK_MCP_SOCKET<br/>(+ proxied plugin tools)"]
    AFM["AFM host bridge"]
    AIB["AiNodeBackend runner"]
  end
  subgraph Services["artifacts / agent"]
    ANALYST[".spur/analyst.duckdb<br/>v_blast_radius, v_symbol_scorecard,<br/>v_file_cochange, duckpgq_*, _meta"]
    GRAPH["code_* tools"]
    ACP["spur-acp NativeAcpConnection"]
    AGENT(["Agent subprocess"])
  end

  Q -- "source.push(question)" --> RE
  RE -- "schedule AI node" --> AIB
  AIB -- "AiRunRequest{prompt, context}" --> AN
  AN -- "AcpAgentBackend.run" --> ACP
  ACP -- "spawn (once) + PromptRequest" --> AGENT
  AGENT -- "call wb_* (notebook MCP socket)" --> MCP
  MCP -- "proxy" --> T1
  MCP -- "proxy" --> T2
  MCP -- "proxy" --> T3
  MCP -- "proxy" --> T4
  T1 -- "read-only SQL" --> ANALYST
  T2 -- "duckpgq + code_*" --> GRAPH
  T3 -- "read-only SQL" --> ANALYST
  T4 -- "read-only SQL" --> ANALYST
  T1 -- "notebook_push_source(analyst_rows, ipc)" --> MCP
  T2 -- "notebook_push_source(subgraph, ipc)" --> MCP
  T3 -- "notebook_push_source(scorecard, ipc)" --> MCP
  T4 -- "notebook_push_source(cochange, ipc)" --> MCP
  MCP -- "push_source" --> RE
  RE -- "deref Arrow" --> PS
  PS --> AFM
  AFM --> G
  AFM --> A
  AFM --> I
  AGENT -- "AgentMessageChunk stream" --> ACP
  ACP -- "answer text" --> AN
  AN -- "answer port" --> AFM
```

**Key integration facts**

1. `notebook_push_source(port, payload: u8[])` = "Push Arrow IPC bytes into a declared source port and queue the reactive engine" — the same `ReactiveEngine::push_source` path a frontend `source.push` uses.
2. Pushed ports must be **declared source ports** (`resolve_source_for_port` rejects undeclared); pushes are debounced per source.
3. The agent sees `wb_*` because the notebook MCP server proxies app-plugin tools additively (foundation tools win on collision).
4. The foundation stays app-agnostic: all workbench logic is in `server/`.

## 3. Reactive DAG (declared source ports)

No Python retrieval cell. The evidence ports are **source ports painted by the plugin tools**; the widgets bind them. `question` is pushed by the composer; `answer` is the AI node output.

```mermaid
flowchart LR
  Q(["question<br/>source port"])
  AN["wb-answer (AI node)<br/>consumes: question<br/>produces: answer"]
  subgraph Tools["app plugin tools (push source ports)"]
    T1["wb_blast_radius -> analyst_rows"]
    T2["wb_subgraph -> subgraph"]
    T3["wb_scorecard -> scorecard"]
    T4["wb_cochange -> cochange"]
  end
  G(["wb-graph<br/>binds: subgraph, scorecard"])
  A(["wb-analyst<br/>binds: analyst_rows"])
  I(["wb-inspector<br/>binds: scorecard, cochange"])
  ANS(["answer (right column)"])

  Q -->|question| AN
  AN -.calls during turn.-> T1
  AN -.calls during turn.-> T2
  AN -.calls during turn.-> T3
  AN -.calls during turn.-> T4
  T2 -->|subgraph| G
  T3 -->|scorecard| G
  T1 -->|analyst_rows| A
  T3 -->|scorecard| I
  T4 -->|cochange| I
  AN -->|answer| ANS
```

The dashed edges are tool invocations inside the agent turn (not DAG edges); the solid edges are port bindings the ReactiveEngine cascades.

## 4. One question turn, end to end

```mermaid
sequenceDiagram
  actor U as User
  participant Q as wb-question
  participant RE as ReactiveEngine
  participant AN as wb-answer (AI node)
  participant ACP as NativeAcpConnection
  participant AG as Agent subprocess
  participant MCP as Notebook MCP (+plugin)
  participant T as wb_* tool
  participant DB as analyst.duckdb
  participant W as Panels (graph/analyst/inspector)

  U->>Q: type question, submit
  Q->>RE: source.push(question)
  RE->>AN: run(AiRunRequest)
  AN->>ACP: ensure_session + prompt
  ACP->>AG: spawn (first turn) + PromptRequest
  AG->>MCP: call wb_blast_radius / wb_subgraph / ...
  MCP->>T: proxy to app plugin
  T->>DB: SELECT FROM v_* (read_only)
  T->>MCP: notebook_push_source(port, arrow ipc)
  MCP->>RE: push_source (queue)
  RE->>W: deref Arrow, render panels
  T-->>AG: result {stable_symbol_ids}
  AG-->>ACP: AgentMessageChunk stream (answer + citations)
  ACP-->>AN: streamed text
  AN->>RE: produce answer port
  RE->>W: render answer (right column)
  Note over U,W: Stop -> conn.cancel(session_id); already-pushed panels stay rendered
```

Panels light up as each `wb_*` tool pushes, mid-turn, before the agent finishes writing the prose answer.

## 5. Backend + plugin model

Real types from `crates/spur-notebook/src/dag/ai/`, `crates/spur-acp/src/connection/native.rs`, and the foundation push path.

```mermaid
classDiagram
  class AiNodeBackend {
    <<trait>>
    +run(AiRunRequest) AiRunOutput
  }
  class AcpAgentBackend {
    -conn: Arc~Mutex~dyn AgentConnection~~
    -cwd: PathBuf
    +new(conn, cwd)
    +ensure_session() SessionId
  }
  class AgentConnection {
    <<trait>>
    +new_session(cwd)
    +prompt(PromptRequest) Stream
    +cancel(session_id)
  }
  class NativeAcpConnection {
    +new(agent_name, command, extra_args, permission_tx)
    +set_repo_root(PathBuf)
    +note broadcast cap 4096 (invariant)
  }
  class AppPluginServer {
    <<FastMCP, server/main.py>>
    +wb_blast_radius(symbol)
    +wb_subgraph(symbol, depth, edge_kinds)
    +wb_scorecard(symbol)
    +wb_cochange(file)
  }
  class notebook_push_source {
    <<foundation MCP tool>>
    +port: String
    +payload: u8[] (Arrow IPC)
  }

  AiNodeBackend <|.. AcpAgentBackend
  AcpAgentBackend --> AgentConnection : conn
  AgentConnection <|.. NativeAcpConnection
  AcpAgentBackend ..> AppPluginServer : agent calls wb_*
  AppPluginServer ..> notebook_push_source : paints ports
```

- **App plugin** mirrors `app_gallery/html_video/server`: `spur-app.json.mcp_server {type python, entry server/main.py}`, FastMCP tools, helper modules. Foundation injects `SPUR_PORTS_ROOT` + `SPUR_NOTEBOOK_MCP_SOCKET` at spawn.
- **Tier-2 later:** swap `conn` for an orchestrator/brain session behind `AiNodeBackend`, "no engine change" (deferred).

## 6. Data mapping: views → tool → port → panel

Every panel is backed by a shipped view (verified live; no invented SQL). The plugin tool is the only writer of each evidence port.

```mermaid
flowchart LR
  subgraph DB[".spur/analyst.duckdb"]
    V1[v_blast_radius]
    V4[v_fix_hotspots]
    V2[v_symbol_scorecard]
    V3[v_file_cochange]
    V5["duckpgq_nodes / duckpgq_edges"]
    V6[onager_edges]
    M[_meta]
  end
  subgraph Tools["app plugin tools"]
    W1[wb_blast_radius]
    W2[wb_subgraph]
    W3[wb_scorecard]
    W4[wb_cochange]
  end
  subgraph Ports["declared source ports"]
    P1[analyst_rows]
    P4[subgraph]
    P2[scorecard]
    P3[cochange]
  end
  subgraph Panels["AFM widgets"]
    PA["analyst (left)"]
    PG["graph (center)"]
    PI["inspector (bottom)"]
  end

  V1 --> W1
  V4 --> W1
  V5 --> W2
  V6 --> W2
  V2 --> W3
  V3 --> W4
  W1 --> P1 --> PA
  W2 --> P4 --> PG
  W3 --> P2 --> PG
  W3 --> P2 --> PI
  W4 --> P3 --> PI
  M -. live counts .-> W1
```

Graph edge-kind chips map 1:1 to `duckpgq_edges.edge_kind` (`calls`, `references_other`, `imports`) plus `onager_edges` (co-change). Index size from `_meta` (~52.8k symbols / ~110k edges), never hardcoded.

## 7. Lifecycle: agent session + plugin server

```mermaid
stateDiagram-v2
  [*] --> Loaded: app opened\nfoundation spawns plugin server (SPUR_PORTS_ROOT, MCP socket)
  Loaded --> Spawning: first question (ensure_session)
  Spawning --> Live: new_session ok\npgid registered (.spur/pgids)
  Live --> Painting: agent calls wb_* -> notebook_push_source -> cascade
  Painting --> Live: panels updated, answer streamed
  Live --> Cancelled: Stop -> conn.cancel(session_id)
  Cancelled --> Live: next question
  Live --> TornDown: app close -> Drop -> killpg(pgid)
  TornDown --> [*]
```

- **Plugin server** is spawned once when the app opens and lives for the session.
- **Agent** session is spawned lazily on the first question and reused (`ensure_session`).
- **Orphan safety:** `NativeAcpConnection` registers the agent pgid under `.spur/pgids/`; `Drop` kills it.

## 8. Boundaries, build items, and invariants

**Trust / failure boundaries**

| Boundary | Rule |
|---|---|
| Widget (sandboxed) → foundation | only via AFM `source.push`; no filesystem, no same-origin |
| Plugin tool → analyst.duckdb | read-only open, path walked up from `cwd`; tolerate transient rebuild locks |
| Agent → services | read-only permission policy (auto-approve reads + `notebook_push_source`, deny writes) |
| Tool / agent failure | isolated: a failed push leaves other panels live; agent failure shows only in the answer column |
| Foundation ↔ app | foundation app-agnostic; plugin tools additive (foundation wins on collision) |

**Critical build items (plan)**

- **Task 0 — API-drift check:** `AcpAgentBackend` / `AiNodeBackend` / `push_source` / `SourcePush` / `notebook_push_source` are young, actively-churning leaves; re-verify against HEAD before building.
- **Task 1 — agent MCP wiring:** the `NativeAcpConnection`-spawned agent must have the notebook MCP socket (foundation + proxied `wb_*` tools) in its config, or it can talk but cannot paint.

**SPUR invariants inherited unchanged**

- Broadcast sizing: `NativeAcpConnection` capacity-4096 channel (anchor `3ff4e86`).
- Orphan teardown: `.spur/pgids/` registry + `Drop` killpg.
- Reactive single-writer cascade: ReactiveEngine schedules; cells never self-fire; bus notify-only; payloads in the Port Store.
- Scripts-off baseline: each widget server-renders last port state.

**Out of scope (see `.md` §10):** Tier-2 brain attach, external-host chat (B2), write actions, multi-repo.